In [ ]:
import sys
sys.path.append("/exp/sbnd/data/users/lnguyen/cafpyana_pi0")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cafpybara.analyses.nuecc as nue

%load_ext autoreload
%autoreload 2

## load dataframes

In [ ]:
dfs_dir = "/exp/sbnd/data/users/lynnt/xsection/samples/MCP2025B_v10_06_00_09/dfs_nu26/"
mcbnb_df, mcbnb_pot, mcbnb_ngen   = nue.load_mc  (dfs_dir+"mc_ar23.df",     keys=['nuecc','hdr'],             cuts=nue.DEFAULT_CUTS)
mclow_df, mclow_pot, mclow_ngen   = nue.load_mc  (dfs_dir+"mc_lowenergy.df",keys=['nuecc','hdr','histpotdf'], cuts=nue.DEFAULT_CUTS)
dtbnb_df, dtbnb_pot, dtbnb_ngates = nue.load_data(dfs_dir+"data_dev.df",    keys=['nuecc','hdr'],onbeam=True, cuts=nue.DEFAULT_CUTS)
dtoff_df, dtoff_pot, dtoff_ngates = nue.load_data(dfs_dir+"offbeamlight.df",keys=['nuecc','hdr'],onbeam=False,cuts=nue.DEFAULT_CUTS)

ongates_per_pot = dtbnb_ngates / dtbnb_pot
f = 0.0725
noffbeamscale_mc = ((1- f)*(ongates_per_pot*mcbnb_pot))/dtoff_ngates
scale = dtbnb_pot/mcbnb_pot

In [ ]:
systs_to_drop = ['GENIEReWeight_SBN_v1_multisigma_ZExpA1CCQE',
                 'GENIEReWeight_SBN_v1_multisigma_ZExpA2CCQE',
                 'GENIEReWeight_SBN_v1_multisigma_ZExpA3CCQE',
                 'GENIEReWeight_SBN_v1_multisigma_ZExpA4CCQE',
                 'ZExpPCAWeighter_SBNNuSyst_multisigma_MvA_ZExp_b1',
                 'ZExpPCAWeighter_SBNNuSyst_multisigma_MvA_ZExp_b2',
                 'ZExpPCAWeighter_SBNNuSyst_multisigma_MvA_ZExp_b3',
                 'ZExpPCAWeighter_SBNNuSyst_multisigma_MvA_ZExp_b4',
                 ]

for dropcol in systs_to_drop:
    if dropcol in list(zip(*list(mcbnb_df.columns)))[2]:  
        mcbnb_df.drop(dropcol,  axis=1,level=2,inplace=True)

In [ ]:
mcbnb_df[('weights_mc', '', '', '', '', '')] =1.0
mclow_df[('weights_mc', '', '', '', '', '')] =mcbnb_pot/mclow_pot
dtoff_df[('weights_mc', '', '', '', '', '')] =noffbeamscale_mc

mcbnb_df[('flux_pot_norm', '', '', '', '', '')] = mcbnb_df.weights_mc/(nue.integrated_flux * (mcbnb_pot / 1e6))
mclow_df[('flux_pot_norm', '', '', '', '', '')] = mclow_df.weights_mc/(nue.integrated_flux * (mcbnb_pot / 1e6))
dtoff_df[('flux_pot_norm', '', '', '', '', '')] = dtoff_df.weights_mc/(nue.integrated_flux * (mcbnb_pot / 1e6))

In [ ]:
mcstat_cols = ['__ntuple','entry','rec.slc..index','run','subrun','evt','sample','file_idx']

mcmc_df = pd.concat([
                        nue.mcstat(mcbnb_df.assign(sample=0).set_index("sample",append=True).reset_index(),cols=mcstat_cols),
                        nue.mcstat(mclow_df.assign(sample=2).set_index("sample",append=True).reset_index(),cols=mcstat_cols),
                        dtoff_df.assign(sample=3).set_index("sample",append=True).reset_index(),
                       ])
data_df = dtbnb_df.copy()

In [ ]:
signal_detvar_file = "/exp/sbnd/data/users/lynnt/xsection/samples/MCP2025B_v10_06_00_09/dfs_nu26/detvars/detvars_signal.h5"
signal_detvar_dict = nue.load_detvar_dict(signal_detvar_file)

signal_systs_cfg = nue.SystematicsInput(
    mcbnb_pot=mcbnb_pot,
    mcbnb_ngen = mcbnb_ngen,
    detvar_dict=signal_detvar_dict,
    # cuts= must match the cuts mcmc_df was loaded with above (DEFAULT_CUTS)
    # -- see README's "Common mistakes" section for why.
    cuts=nue.DEFAULT_CUTS)

signal_plots_cfg = nue.PlottingConfig(
    scale=scale,
    ylabel=f"Events [{dtbnb_pot:.2e} POT]",
    systs=signal_systs_cfg,
    percents=True,
    ratio_min=0.,
    ratio_max=2,
    title=" ")

## make plots

In [ ]:
output = nue.plot_mc_data(
                 mc_df   = mcmc_df,
                 data_df = data_df,
                 var=nue.electron_direction().var_evt_reco_col,
                 bins=nue.electron_direction().bins,
                 bin_labels=nue.electron_direction().bin_labels,
                 xlabel="Electron Direction, $\\cos\\theta$",
                 legend_kwargs={'loc':'upper left','ncol':2,'fontsize':9},
                 config=signal_plots_cfg)
plt.show()

output = nue.plot_mc_data(
                 mc_df=mcmc_df,
                 data_df = data_df,
                 var=nue.electron_energy().var_evt_reco_col,
                 bins=nue.electron_energy().bins,
                 bin_labels=nue.electron_energy().bin_labels,
                 xlabel="Electron Energy [GeV]",
                 legend_kwargs={'loc':'upper right','ncol':2,'fontsize':9},
                 config=signal_plots_cfg)
plt.show()

## get syst breakdown

In [ ]:

signal_energy_output = nue.get_total_cov(
    mcmc_df,
    reco_var=nue.electron_energy().var_evt_reco_col,
    bins=nue.electron_energy().bins,
    **signal_systs_cfg.to_kwargs()
    )
signal_direct_output = nue.get_total_cov(
    mcmc_df,
    reco_var=nue.electron_direction().var_evt_reco_col,
    bins=nue.electron_direction().bins,
    **signal_systs_cfg.to_kwargs()
    )

In [ ]:
_dir_var = nue.electron_direction()
_ene_var = nue.electron_energy()
syst_vars = [
    (signal_direct_output, _dir_var.bins, r"Electron Direction, $\cos\theta$", _dir_var.bin_labels),
    (signal_energy_output,  _ene_var.bins,  "Electron Energy [GeV]",              _ene_var.bin_labels),
]

fig, axes, cats, cat_sums = nue.plot_syst_category_breakdown(
    syst_vars=syst_vars,
    category_dict=nue.category_dict_signal,
)
plt.show()

In [ ]:
fig, axes = nue.plot_syst_breakdown(
    syst_vars=syst_vars,
    category='DetVar',
    category_dict=nue.category_dict_signal,
)
plt.show()

In [ ]:
fig, axes = nue.plot_syst_breakdown(
    syst_vars=syst_vars,
    category='GENIE',
    category_dict=nue.category_dict_signal,
)
plt.show()